# Aegis Finance - Phase 1: Data Foundation Showcase

This notebook demonstrates the foundational data layer built during Phase 1. 

**Goals of Phase 1:**
1. Database connectivity (`PostgreSQL` + `pgvector`).
2. Synthetic data generation.
3. Statement parser framework.
4. Transaction categorization (Rule-based baseline).


In [ ]:
import os
import sys
from pathlib import Path
from decimal import Decimal
import pandas as pd

# Setup paths so we can import from src
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

# Load environment configuration
from aegis.config import settings
print(f"Loaded config for env: {settings.env}")


## 1. Database Connectivity

We use `psycopg` connection pools. Let's verify our connection to PostgreSQL.


In [ ]:
from aegis.db.connection import get_db_pool

pool = get_db_pool()

with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        version = cur.fetchone()[0]
        print(f"Connected to: {version}")


## 2. Synthetic Data Overview

Phase 1 included `data/synthetic/generate.py` to seed our local environment with realistic Argentine financial data (accounts, USD/ARS transactions, crypto, and market rates). Let's query some views from `002_views.sql`.


In [ ]:
with pool.connection() as conn:
    # Let's use pandas to nicely format the SQL outputs
    df_accounts = pd.read_sql("SELECT * FROM v_current_balances;", conn)
    
print("Current Balances (v_current_balances):")
display(df_accounts)


In [ ]:
with pool.connection() as conn:
    df_monthly = pd.read_sql("SELECT * FROM v_monthly_spending ORDER BY month DESC LIMIT 5;", conn)

print("Recent Monthly Spending (v_monthly_spending):")
display(df_monthly)


In [ ]:
with pool.connection() as conn:
    df_rates = pd.read_sql("SELECT * FROM v_latest_exchange_rates;", conn)

print("Latest Exchange Rates (v_latest_exchange_rates):")
display(df_rates)


## 3. Parsing & Categorization Framework

Aegis Finance provides an extensible parsing framework. Here we demonstrate parsing a sample CSV and running it through the rule-based categorizer.


In [ ]:
from aegis.parsers.bank_csv import BankCSVParser
from aegis.parsers.categorizer import RuleBasedCategorizer

# Create a sample CSV mimicking a bank statement
sample_csv_path = project_root / "tests" / "fixtures" / "sample_bank.csv"

# Make sure the fixture exists
if not sample_csv_path.exists():
    sample_csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(sample_csv_path, "w", encoding="utf-8") as f:
        f.write("FECHA,SUCURSAL,DESCRIPCION,REFERENCIA,IMPORTE,SALDO\n")
        f.write("01/03/2026,000,CARREFOUR SUC 42,, -45000.00, 100000.00\n")
        f.write("02/03/2026,000,UBER *TRIP,, -2500.00, 97500.00\n")
        f.write("05/03/2026,000,ACREDITACION HABERES - TECH S.A.,, 950000.00, 1047500.00\n")
        f.write("06/03/2026,000,COMPRA DOLAR MEP - GALICIA,, -500000.00, 547500.00\n")
        f.write("10/03/2026,000,UNKNOWN STORE BA,, -15000.00, 532500.00\n")

# 1. Parse the CSV
# Note: In practice, account_id comes from the user context
import uuid
dummy_account_id = uuid.uuid4()

parser = BankCSVParser(
    account_id=dummy_account_id,
    currency="ARS",
    date_col="FECHA",
    amount_col="IMPORTE",
    desc_col="DESCRIPCION",
    date_format="%d/%m/%Y"
)

transactions = parser.parse(sample_csv_path)

print(f"Parsed {len(transactions)} transactions.")


In [ ]:
# 2. Categorize the transactions
categorizer = RuleBasedCategorizer()
categorized_txns = categorizer.categorize(transactions)

# Show the results
records = []
for t in categorized_txns:
    records.append({
        "Date": t.date,
        "Raw Merchant": t.merchant_raw,
        "Amount": t.amount,
        "Category": t.category,
        "Score": t.category_score,
        "Flagged (HITL)": t.is_flagged
    })

df_parsed = pd.DataFrame(records)
display(df_parsed)
